In [329]:
import torch
from torch import nn
import torch.optim as optim

import time
from tqdm import tqdm

import numpy as np
from text_helpers import load_corpus, Vocab, batch_generator, encode_text, decode_text, generate_text, sample_from_probs, generate_seed

In [330]:
# setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


In [331]:
rnn = nn.RNN(10, 20, 2)
input = torch.randn(5, 3, 10)
h0 = torch.randn(2, 3, 20)
output, hn = rnn(input, h0)

In [332]:
def build_rnn(vocab_size, embedding_size, hidden_size, num_layers):
    
    def init_weights(m):
        if type(m) == torch.nn.Linear:
            torch.nn.init.xavier_uniform_(m.weight)
            m.bias.data.fill_(0.01)
    
    # output shape: [seq_len, batch_size, hidden_size]
    
    class CharRNN(nn.Module):
        def __init__(self, vocab_size, embedding_size, hidden_size, num_layers):
            super(CharRNN, self).__init__()
            self.hidden_size = hidden_size
            self.num_layers = num_layers
            
            self.embedding = nn.Embedding(vocab_size, embedding_size)
            self.embedding_dropout = nn.Dropout(0.3)

            self.rnn = nn.RNN(embedding_size, hidden_size, num_layers, batch_first=True, dropout=0.4 if num_layers > 1 else 0)
            self.rnn_dropout = nn.Dropout(0.3)

            self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
            self.relu = nn.ReLU()
            self.dropout_fc = nn.Dropout(0.3)
            
            self.fc2 = nn.Linear(hidden_size // 2, hidden_size // 4)
            self.relu = nn.ReLU()
            
            self.fc3 = nn.Linear(hidden_size // 4, vocab_size)
            
        def forward(self, x, hidden=None):
            # x shape: [batch_size, seq_len]
            batch_size = x.size(0)
            
            # embedding: [batch_size, seq_len, embedding_size]
            x = self.embedding(x)
            x = self.embedding_dropout(x)
            
            # rnn output: [batch_size, seq_len, hidden_size]
            if hidden is None:
                hidden = self.init_hidden(batch_size)
            
            output, hidden = self.rnn(x, hidden)
            output = self.rnn_dropout(output)
            
            # проходим через полносвязные слои
            output = self.fc1(output)
            output = self.relu(output)
            output = self.dropout_fc(output)
            output = self.fc2(output)
            output = self.relu(output)
            output = self.fc3(output)
            
            return output, hidden
        
        def init_hidden(self, batch_size):
            device = next(self.parameters()).device
            return torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
    
    net = CharRNN(vocab_size, embedding_size, hidden_size, num_layers)
    net.apply(init_weights)
    return net

In [333]:
def train(net, train_loader, device, num_epochs, learning_rate, num_batches, corpus, vocab):
    
    optimizer = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5)
    loss_function = torch.nn.CrossEntropyLoss()
    loss_history = []

    with tqdm(total=num_batches*num_epochs, position=0, leave=True) as pbar:
        net = net.to(device)
        
        for epoch in range(num_epochs):
            net.train()
            running_loss = 0.0
            
            for batch_idx in range(num_batches):
                (inputs, labels, *_) = next(train_loader)
                inputs = torch.from_numpy(inputs).to(device).long()
                labels = torch.from_numpy(labels).to(device).long()
                
                optimizer.zero_grad()
                
                # Forward pass
                output, _ = net(inputs)
                
                # Reshape for loss: [batch_size * seq_len, vocab_size]
                output = output.reshape(-1, output.size(-1))
                labels = labels.reshape(-1)
                
                loss = loss_function(output, labels)
                
                # Backpropagation
                loss.backward()
                
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                
                # Update
                optimizer.step()
                
                # Print progress
                running_loss += loss.item()
                pbar.update(1)
            
            avg_loss = running_loss / num_batches
            scheduler.step(avg_loss)
            
            loss_history.append(avg_loss)
            
            # Generate text every few epochs
            if epoch % 20 == 0:
                seed = generate_seed(corpus)
                print(f"\nEpoch {epoch}, Loss: {avg_loss:.4f}")
                generate_text(net, seed, vocab=vocab, device=device)

    pbar.close()
    return loss_history

In [334]:
def generate_text(model, seed, length=100, top_n=5, vocab=None, device='cpu'):
    import torch
    model.eval()
    
    print(f"\nGenerating {length} characters with seed: \"{''.join(seed)}\"")
    
    generated = list(seed)
    encoded = encode_text(seed, char2id=vocab)
    input_tensor = torch.tensor(encoded).unsqueeze(0).to(device).long()  # [1, seq_len]
    
    hidden = None
    
    for i in range(length):
        # Forward pass
        with torch.no_grad():
            output, hidden = model(input_tensor, hidden)
            
        # Get last output
        probs = torch.softmax(output[0, -1], dim=-1)
        probs = probs.cpu().numpy()
        
        # Sample from top_n choices
        top_indices = np.argsort(probs)[-top_n:]
        top_probs = probs[top_indices]
        top_probs = top_probs / top_probs.sum()
        
        next_index = np.random.choice(top_indices, p=top_probs)
        
        # Append to sequence
        next_char = vocab.to_tokens(next_index)
        generated.append(next_char)
        
        # Update input for next step
        input_tensor = torch.tensor([[next_index]]).to(device).long()
    
    generated_text = ''.join(generated)
    print(f"Generated text:\n{generated_text}\n")
    return generated_text

In [335]:
# read text
corpus, vocab = load_corpus('data/tinyshakespeare.txt', token_type = 'char')
VOCAB_SIZE = len(vocab)

In [336]:
# print vocab index

vocab._token_to_idx

{'<unk>': 0,
 ' ': 1,
 'e': 2,
 't': 3,
 'o': 4,
 'a': 5,
 'h': 6,
 's': 7,
 'r': 8,
 'n': 9,
 'i': 10,
 '\n': 11,
 'l': 12,
 'd': 13,
 'u': 14,
 'm': 15,
 'y': 16,
 ',': 17,
 'w': 18,
 'f': 19,
 'c': 20,
 'g': 21,
 'I': 22,
 'b': 23,
 'p': 24,
 ':': 25,
 '.': 26,
 'A': 27,
 'v': 28,
 'k': 29,
 'T': 30,
 "'": 31,
 'E': 32,
 'O': 33,
 'N': 34,
 'R': 35,
 'S': 36,
 'L': 37,
 'C': 38,
 ';': 39,
 'W': 40,
 'U': 41,
 'H': 42,
 'M': 43,
 'B': 44,
 '?': 45,
 'G': 46,
 '!': 47,
 'D': 48,
 '-': 49,
 'F': 50,
 'Y': 51,
 'P': 52,
 'K': 53,
 'V': 54,
 'j': 55,
 'q': 56,
 'x': 57,
 'z': 58,
 'J': 59,
 'Q': 60,
 'Z': 61,
 'X': 62,
 '3': 63,
 '&': 64,
 '$': 65}

In [337]:
EMMBEDDING_SIZE = 32
HIDDEN_SIZE = 128
NUM_LAYERS = 2

model = build_rnn(VOCAB_SIZE, EMMBEDDING_SIZE, HIDDEN_SIZE, NUM_LAYERS)

In [338]:
print(model)

CharRNN(
  (embedding): Embedding(66, 32)
  (embedding_dropout): Dropout(p=0.3, inplace=False)
  (rnn): RNN(32, 128, num_layers=2, batch_first=True, dropout=0.4)
  (rnn_dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (dropout_fc): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=66, bias=True)
)


In [339]:
BATCH_SIZE = 16
SEQ_LENGTH = 32
EPOCHS = 50 
LR = 0.0001

num_batches = (len(corpus) - 1) // (BATCH_SIZE * SEQ_LENGTH)
data_iter = batch_generator(encode_text(corpus, char2id=vocab), batch_size = BATCH_SIZE, seq_len=SEQ_LENGTH, vocab = vocab)

In [340]:
loss_history = train(model, data_iter, device, EPOCHS, LR, num_batches, corpus, vocab)

  0%|          | 1/108900 [00:00<5:39:28,  5.35it/s]

number of batches: 2178.
effective text length: 1115136.
x shape:  (16, 69696)
y shape:  (16, 69696)


  2%|▏         | 2185/108900 [00:36<33:19, 53.36it/s]


Epoch 0, Loss: 2.9926

Generating 100 characters with seed: " b"
Generated text:
 beat,,
To tor and wet thar tane wate thire so thore sond
Th thit, sher thing tolt to mor hin ant we t



 42%|████▏     | 45742/108900 [12:37<19:17, 54.57it/s]


Epoch 20, Loss: 2.1733

Generating 100 characters with seed: " Pompey; and, in requital of you"
Generated text:
 Pompey; and, in requital of you him thy mone the that.
Whing to so steal are ale, the moulds have shar shall
To the sir herars, wha



 82%|████████▏ | 89306/108900 [24:23<06:05, 53.64it/s]


Epoch 40, Loss: 2.1009

Generating 100 characters with seed: "rince,
Jewel of children, seen t"
Generated text:
rince,
Jewel of children, seen thou this whome weak where. Would stow they wangen and here home hings, the ster his weake the sear t



100%|██████████| 108900/108900 [29:41<00:00, 61.13it/s]


In [341]:
seed = generate_seed(corpus)
generate_text(model, seed, length=200, top_n=5, vocab=vocab, device=device)
encoded = encode_text(seed, char2id=vocab)


Generating 200 characters with seed: " or mistaking:
t"
Generated text:
 or mistaking:
to she wilts word. What sor seld with thy to whom seer,
The well son would the tear she his store, strough him this
To sir and his sheak that sist the sis the wicks,
Tith sead that and why will and a t



In [342]:
print (encoded)

[ 1  4  8  1 15 10  7  3  5 29 10  9 21 25 11  3]


In [343]:
tensor = torch.tensor(encoded).to(device)

In [344]:
tensor

tensor([ 1,  4,  8,  1, 15, 10,  7,  3,  5, 29, 10,  9, 21, 25, 11,  3])

In [346]:
input_tensor = tensor.unsqueeze(0)  # [1, seq_len] - batch_size=1
output, hidden = model(input_tensor)

In [347]:
output.shape
# torch.softmax(output, -2)

torch.Size([1, 16, 66])

In [348]:
last_output = output[-1, :] 
probs = torch.softmax(last_output, dim=0) 
probs_np = probs.detach().cpu().numpy()
sampled_index = sample_from_probs(probs_np, top_n=5)

In [349]:
# model, seed, length=512, top_n=10, vocab=Vocab
generate_text(model, "KING RICHARD: ", length=512, top_n=4, vocab=vocab)


Generating 512 characters with seed: "KING RICHARD: "
Generated text:
KING RICHARD: Where ald tould thees ale hound, then the wolt they him the sear heater thing a me seen some here, sinest that to the hould and seast this were and sore the well store a serdens son see and that thee tread they the worth hears,
Which shall, that he to thou then.

CRELEN:
The so whish heart thou whom thou trist his the world the shere have then a strence and then his the this him senger, shall and the well,
As the so that to thene
Iffond the that that heathens, ale,
What are.

GORT HERRINCA:
Whan the trine s



'KING RICHARD: Where ald tould thees ale hound, then the wolt they him the sear heater thing a me seen some here, sinest that to the hould and seast this were and sore the well store a serdens son see and that thee tread they the worth hears,\nWhich shall, that he to thou then.\n\nCRELEN:\nThe so whish heart thou whom thou trist his the world the shere have then a strence and then his the this him senger, shall and the well,\nAs the so that to thene\nIffond the that that heathens, ale,\nWhat are.\n\nGORT HERRINCA:\nWhan the trine s'